# KBO 선수 트래킹

특정 선수의 커리어 스탯 변화, 동 포지션 대비 순위 추이를 시각화합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# 한글 폰트 설정
try:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumGothic'
except:
    try:
        plt.rcParams['font.family'] = 'Malgun Gothic'
    except:
        pass

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

In [ ]:
# 데이터 로드
def load_all_seasons():
    files = sorted(RAW_DIR.glob('kbo_*.csv'))
    if not files:
        raise FileNotFoundError('데이터 없음. src/crawl.py 먼저 실행하세요.')
    import re
    frames = []
    for f in files:
        df = pd.read_csv(f, dtype=str)
        # Name 컬럼에서 한글 제거 (영문명만 추출)
        df['Name'] = df['Name'].str.replace(r'[\u0e00-\u9fff\uac00-\ud7af]+', '', regex=True).str.strip()
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)

    skip = {'Name', 'Team', 'PlayerName', 'KName'}
    num_cols = [c for c in df.columns if c not in skip]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df = df[df['PA'] >= 50].copy()
    return df

df = load_all_seasons()
print(f'총 {len(df)}행, {df["Season"].min():.0f}~{df["Season"].max():.0f}시즌')
print(f'컬럼: {list(df.columns)}')

In [ ]:
# 선수 검색
def search_player(query: str) -> pd.DataFrame:
    return df[df['Name'].str.contains(query, case=False, na=False)][['Name','Season','Team','PA','AVG','OBP','SLG','wRC+','WAR']].drop_duplicates()

# 예시: 검색
search_player('Park')  # 이름으로 검색 (영문)

In [ ]:
# ── 선수 커리어 대시보드 ──────────────────────────────────────────────────────

def player_dashboard(name: str):
    """선수 이름(영문) 입력 → 커리어 스탯 + 리그 대비 순위 시각화."""
    p = df[df['Name'].str.contains(name, case=False, na=False)].sort_values('Season')
    if p.empty:
        print(f'"{name}" 선수를 찾을 수 없습니다. search_player()로 검색해 보세요.')
        return

    display_name = p['Name'].iloc[0]
    seasons = p['Season'].values

    # 리그 퍼센타일 계산
    def pct_rank(col):
        ranks = []
        for _, row in p.iterrows():
            season_df = df[df['Season'] == row['Season']]
            val = row[col]
            if pd.isna(val): ranks.append(np.nan); continue
            pct = (season_df[col].dropna() < val).mean() * 100
            ranks.append(pct)
        return ranks

    metrics = ['wRC+', 'AVG', 'OBP', 'SLG', 'WAR']
    colors  = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(f'{display_name} 커리어 트래킹', fontsize=16, fontweight='bold')

    # 1) 주요 스탯 raw 값
    ax = axes[0, 0]
    for m, c in zip(['wRC+'], ['#2196F3']):
        if m in p.columns:
            ax.plot(seasons, p[m], marker='o', color=c, linewidth=2.5, markersize=7, label=m)
            ax.axhline(100, color='gray', linestyle='--', alpha=0.5, label='리그 평균(100)')
    ax.set_title('wRC+ 추이'); ax.set_xlabel('시즌'); ax.legend()
    ax.grid(alpha=0.3)

    # 2) 타율/출루율/장타율
    ax = axes[0, 1]
    for m, c in zip(['AVG', 'OBP', 'SLG'], ['#4CAF50', '#FF9800', '#E91E63']):
        if m in p.columns:
            ax.plot(seasons, p[m], marker='o', linewidth=2.5, markersize=7, label=m, color=c)
    ax.set_title('타율/출루율/장타율'); ax.set_xlabel('시즌'); ax.legend()
    ax.grid(alpha=0.3)

    # 3) WAR 추이
    ax = axes[1, 0]
    if 'WAR' in p.columns:
        ax.bar(seasons, p['WAR'], color='#9C27B0', alpha=0.8)
        ax.set_title('WAR 추이'); ax.set_xlabel('시즌'); ax.set_ylabel('WAR')
        ax.grid(alpha=0.3, axis='y')

    # 4) PA 추이
    ax = axes[1, 1]
    if 'PA' in p.columns:
        ax.bar(seasons, p['PA'], color='#607D8B', alpha=0.8)
        ax.set_title('타석 수 (PA)'); ax.set_xlabel('시즌'); ax.set_ylabel('PA')
        ax.grid(alpha=0.3, axis='y')

    # 5) 리그 내 퍼센타일 (wRC+)
    ax = axes[2, 0]
    pct = pct_rank('wRC+')
    bar_colors = ['#F44336' if v >= 90 else '#2196F3' if v >= 50 else '#9E9E9E' for v in [v if not np.isnan(v) else 0 for v in pct]]
    ax.bar(seasons, pct, color=bar_colors, alpha=0.85)
    ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim(0, 100); ax.set_title('wRC+ 리그 내 퍼센타일'); ax.set_xlabel('시즌'); ax.set_ylabel('%')
    ax.grid(alpha=0.3, axis='y')

    # 6) 스탯 테이블
    ax = axes[2, 1]
    ax.axis('off')
    show_cols = ['Season', 'Team', 'PA', 'AVG', 'OBP', 'SLG', 'wRC+', 'WAR']
    show_cols = [c for c in show_cols if c in p.columns]
    tbl_data = p[show_cols].tail(8).values.tolist()
    tbl_data = [[f'{v:.3f}' if isinstance(v, float) and 0 < abs(v) < 10 else
                 f'{v:.0f}' if isinstance(v, float) else str(v) for v in row]
                for row in tbl_data]
    tbl = ax.table(cellText=tbl_data, colLabels=show_cols, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
    tbl.auto_set_column_width(col=list(range(len(show_cols))))
    ax.set_title('시즌별 스탯 요약')

    plt.tight_layout()
    plt.savefig(f'../data/{display_name.replace(" ","_")}_dashboard.png', bbox_inches='tight')
    plt.show()
    print(f'저장: data/{display_name.replace(" ","_")}_dashboard.png')

# 사용 예시 — 이름을 원하는 선수로 바꾸세요
# player_dashboard('Park')

In [ ]:
# ── 동 포지션 / 시즌 top-N 비교 ──────────────────────────────────────────────

def compare_with_peers(name: str, season: int = None, top_n: int = 10, metric: str = 'wRC+'):
    """해당 시즌 상위 선수들과 레이더 차트 비교."""
    p = df[df['Name'].str.contains(name, case=False, na=False)]
    if p.empty:
        print('선수 없음'); return
    if season is None:
        season = int(p['Season'].max())

    season_df = df[(df['Season'] == season) & (df['PA'] >= 200)].copy()
    player_row = season_df[season_df['Name'].str.contains(name, case=False, na=False)]

    radar_cols = ['AVG', 'OBP', 'SLG', 'wRC+', 'WAR', 'BB%', 'K%']
    radar_cols = [c for c in radar_cols if c in season_df.columns]

    top = season_df.nlargest(top_n, metric)
    if player_row.empty:
        print(f'{season}시즌에 {name} 데이터 없음')
        return

    # 정규화 (0~1)
    norm = season_df[radar_cols].copy()
    for c in radar_cols:
        mn, mx = norm[c].min(), norm[c].max()
        norm[c] = (norm[c] - mn) / (mx - mn + 1e-9)
        if c in ['K%']:  # K%는 낮을수록 좋음
            norm[c] = 1 - norm[c]

    # 레이더 차트
    angles = np.linspace(0, 2*np.pi, len(radar_cols), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    # 평균선
    avg_vals = norm.mean()[radar_cols].tolist() + [norm.mean()[radar_cols[0]]]
    ax.plot(angles, avg_vals, color='gray', linewidth=1, linestyle='--', label='리그 평균')
    ax.fill(angles, avg_vals, alpha=0.05, color='gray')

    # 선수
    idx = player_row.index[0]
    p_vals = norm.loc[idx, radar_cols].tolist() + [norm.loc[idx, radar_cols[0]]]
    display_name = player_row['Name'].iloc[0]
    ax.plot(angles, p_vals, color='#2196F3', linewidth=2.5, label=display_name)
    ax.fill(angles, p_vals, alpha=0.2, color='#2196F3')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_cols, fontsize=11)
    ax.set_title(f'{display_name} vs 리그 ({season}시즌)', fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()

# 사용 예시
# compare_with_peers('Park', season=2023)

In [ ]:
# ── 실제 분석 실행 ─────────────────────────────────────────────────────────────
# 아래 이름/시즌을 바꿔서 실행하세요

# 1) 어떤 선수가 있는지 확인
print('=== 전체 선수 목록 (2023시즌 PA 200+) ===')
season_sample = df[(df['Season'] == 2023) & (df['PA'] >= 200)][['Name', 'Team', 'PA', 'wRC+', 'WAR']]
print(season_sample.sort_values('wRC+', ascending=False).to_string(index=False))